In [ ]:
import sqlite3 as sql
import pandas as pd
from IPython.display import display

##Conection and exploration

In [ ]:
connect = sql.connect('games.db') #Conecta ao arquivo do banco ou cria o arquivo
print(f'connection successfully established') 

df_game_raw = pd.read_csv('Video_Games.csv') #Carrega o dataset 

In [ ]:
#Verificando tamanho e amostra
print(f'{df_game_raw.shape}') #Quantas linhas e colunas 
#df_game_raw.head() #Mostra 5 primeiros

In [ ]:
#Verificando se há coisas faltando

print(f'{df_game_raw['index'].is_unique} \n') #Verifico se o index é único

print(f'Empty values from CriticScore: {df_game_raw['Critic_Score'].isnull().sum()}\n ') #Confere o total de valores vazios na coluna

print(f'Empty values from User Score: {df_game_raw['User_Score'].isnull().sum()} \n ')

print(f'Untitle Games: {df_game_raw['Name'].isnull().sum()}\n ')

print(f'Games without a release year: {df_game_raw['Year_of_Release'].isnull().sum()}\n ')

print(f'Games without a publisher: {df_game_raw['Publisher'].isnull().sum()}')

In [ ]:
#Tratamento das colunas

df_game_raw['Critic_Score'] = pd.to_numeric( #Tratando para virar númerico, se não vira NaN
    df_game_raw['Critic_Score'],
    errors = 'coerce'
)

df_game_raw['User_Score'] = pd.to_numeric(
    df_game_raw['User_Score'], 
    errors = 'coerce'
) 

df_game_raw['Year_of_Release'] = df_game_raw['Year_of_Release'].astype('Int64') #Tratando os números que estavam 2000.0 -> 2000

df_game_raw['Name'] = df_game_raw['Name'].fillna('Unknown') #Preenchendo valores vazios por Unknown
df_game_raw['Publisher'] = df_game_raw['Publisher'].fillna('Unknown')
df_game_raw['Rating'] = df_game_raw['Rating'].fillna('Unknown')

In [ ]:
print(f'The year data type is: {df_game_raw['Year_of_Release'].dtype}\n ')

print(f'Games still without a name: {df_game_raw['Name'].isnull().sum()} \n')

print(f'Games without a publisher yet: {df_game_raw['Publisher'].isnull().sum()} \n')

print(f'Games without an age rating yet: {df_game_raw['Rating'].isnull().sum()} \n')

In [ ]:
#Criando o database
with open ('schema.sql', 'r', encoding = 'utf-8') as archive: #Abre o arquivo em modo leitura 'r = read'
    command_sql = archive.read()

cursor = connect.cursor() #Crio quem envia comandos ao banco

cursor.executescript(command_sql) #Executo o sql e salva alterações
connect.commit()

print(f'Tables successfully created')

In [ ]:
publisher_unique = df_game_raw['Publisher'].unique() 

df_publisher = pd.DataFrame({
    'publisher_id': range(1, len(publisher_unique) +1), #Criando o id de publsiher
    'publisher_name': publisher_unique
})

df_cross = df_game_raw.merge(df_publisher, left_on = 'Publisher', right_on = 'publisher_name', how = 'left')

df_game_normalized = df_cross[['index', 'Name', 'Year_of_Release', 'Platform', 'Genre','publisher_id']].copy() #Recorta apenas as colunas que vão para o banco
df_game_normalized.columns = ['game_id','name', 'year', 'platform', 'genre', 'publisher_id'] #Renomeia de acordo com o schema.sql

df_info = df_cross[['index', 'Global_Sales', 'Critic_Score', 'User_Score', 'Rating']].copy()
df_info.columns = ['game_id', 'global_sale_millions', 'critic_score', 'user_score', 'rating']
df_info.insert(0, 'info_id', range(1, len(df_info) +1))

In [ ]:
print(f'Sample: ')
display(df_publisher.head(5))

print(f'Samples game:')
display(df_game_normalized.head(5))

print(f'Samples info:') 
display(df_info.head())

In [ ]:
df_publisher.to_sql('publisher', connect, if_exists = 'append', index = False)  #Envio tabela para o sql 
print(f'The table publisher as created')

df_game_normalized.to_sql('game', connect, if_exists = 'append', index = False) 
print(f'The table game as created')

df_info.to_sql('info', connect, if_exists = 'append', index = False)
print(f'The table info as created')

connect.close()
print(f'Connection closed')

##Analysis

In [ ]:
with sql.connect('games.db') as connect:
    df_top_games = pd.read_sql_query(""" 
    SELECT game.name AS Game,
        publisher.publisher_name AS Publisher,
        info.global_sale_millions AS Millions_Copies_Sold

    FROM game
    JOIN publisher ON game.publisher_id = publisher.publisher_id 
    JOIN info ON game.game_id = info.game_id 
    ORDER BY info.global_sale_millions DESC
    LIMIT 10;
    """, connect)

    display(df_top_games)

In [ ]:
with sql.connect('games.db') as connect:
    df_avg = pd.read_sql_query("""
        SELECT 
            CAST((AVG(info.global_sale_millions) * 1000000) AS INT) AS Average_Sales_Units
        FROM info;
        """, connect)
    display(df_avg)